<a href="https://colab.research.google.com/github/GiovanniPasq/agentic-rag-for-dummies/blob/main/notebooks/pdf_to_markdown.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 面向 RAG 系统的 PDF 转 Markdown

## 引言

在构建高质量 RAG（Retrieval-Augmented Generation）系统时，将 PDF 转为 Markdown 往往是**最关键的一步**。Markdown 在结构保真与处理成本之间取得了良好平衡：既能保留原文档层级（标题、列表、表格、代码、公式），又足够轻量，LLM 可直接消费而无需复杂预处理。

**为什么选择 Markdown，而不是纯文本或 JSON？**

虽然纯文本提取更快、JSON 结构化更强，但对 RAG 而言，**Markdown 更合适**，原因包括：

- **保留文档结构**：标题、子标题、列表与格式层级可完整保留，有助于 LLM 理解章节逻辑与语义关系
- **保留语义线索**：粗体、斜体、代码块等格式信息可提供额外上下文
- **自然表示复杂元素**：表格、代码块、数学公式、引用块都有统一且可读的表示方式
- **兼顾人类与机器可读性**：相比 JSON 的刚性键值结构与纯文本的无层级，Markdown 更平衡
- **更适合分块**：结构边界清晰（标题、段落），便于做高质量检索分块

纯文本会丢失大量格式与结构，导致检索系统难以区分标题、正文与元数据。JSON 能表达结构，但在文本密集文档中通常冗长且依赖自定义 schema。Markdown 自然地填补了两者之间的空白。

**为什么这一步如此重要？**

RAG 系统的上限由抽取数据质量决定。若抽取结果“脏”或不完整（格式破碎、表格缺失、公式乱码、上下文丢失），会直接导致检索偏差和回答幻觉。设计抽取流程前，建议先回答：

- **你的 PDF 内容类型是什么？** 仅文本？包含图片？表格？数学公式？
- **版式复杂度如何？** 单栏？多栏？是否混合侧边栏？
- **是否有关键视觉元素？** 图表、示意图、照片是否承载核心语义？
- **是否为扫描件？** 扫描 PDF 无论布局简单与否都需要 OCR。

基于这些问题，可将 PDF 分为三个复杂度层级：

---

## PDF 复杂度分级

**🟢 简单 PDF（类别 1）**
- 纯文本为主、标准版式
- 数字原生 PDF（文本可选中）
- 示例：报告、文章、普通书籍、文档
- **注意：** 若是扫描件，应归入类别 2（需 OCR）

**🟡 中等复杂 PDF（类别 2）**
- 含表格和基础格式
- 扫描件（即便版式简单）
- 偶尔包含图片
- 多栏布局
- 示例：学术论文、商业报告、扫描书籍

**🔴 高复杂 PDF（类别 3）**
- 图像占比高且视觉信息关键
- 复杂图表、流程图、信息图
- 多种内容混排且存在空间关系
- 示例：含图示的科研论文、医疗报告、技术手册、演示文稿

---

## 🐿️ Chunky：验证你的 RAG 数据

为了可视化测试本 notebook 中的方法，你可以使用 **[Chunky](https://github.com/GiovanniPasq/chunky)**：一款本地工具，用于**可视化、编辑和验证** PDF 到 Markdown 的转换与分块效果。

<p align="center">
  <img src="https://raw.githubusercontent.com/GiovanniPasq/chunky/main/assets/demo.gif" width="650">
</p>

- **Compare**：一键切换 PyMuPDF、Docling 与 VLM 结果。
- **Inspect**：将分块与原始 PDF 并排查看。
- **Fix**：手工修正转换伪影与错误分块。

---

## 类别 1：简单 PDF - 快速文本提取

**适用场景：** PDF 为数字原生（非扫描）、以文本为主、格式简单且无关键视觉元素。

**推荐工具：**
- **PyMuPDF4LLM** - https://github.com/pymupdf/PyMuPDF4LLM
- **文档** - https://pymupdf.readthedocs.io/en/latest/pymupdf4llm/index.html

### 示例实现：PyMuPDF4LLM

In [ ]:
!pip install pymupdf4llm

In [ ]:
import pymupdf4llm
import pathlib

def convert_simple_pdfs_pymupdf4llm(pdf_folder: str, output_folder: str):
    """
    Convert simple text-based PDFs to Markdown using PyMuPDF4LLM.

    Args:
        pdf_folder: Path to folder containing PDF files
        output_folder: Path to output folder for Markdown files
    """
    pdf_path = pathlib.Path(pdf_folder)
    output_path = pathlib.Path(output_folder)
    output_path.mkdir(parents=True, exist_ok=True)

    for pdf_file in pdf_path.glob("*.pdf"):
        try:
            # Extract text as Markdown
            md_text = pymupdf4llm.to_markdown(str(pdf_file))

            # Save to file
            output_file = output_path / f"{pdf_file.stem}.md"
            output_file.write_text(md_text, encoding='utf-8')

            print(f"✓ Converted: {pdf_file.name}")

        except Exception as e:
            print(f"✗ Error processing {pdf_file.name}: {e}")

    print(f"\nConversion complete! Output in '{output_folder}'")

# Example usage
convert_simple_pdfs_pymupdf4llm("./simple_pdfs", "./md_output")

---

## 类别 2：中等复杂 PDF - OCR + 结构识别

**适用场景：** 文档为扫描件、需要保留表格结构，或存在多栏布局。

**推荐工具：**
- **Docling** - https://github.com/DS4SD/docling
- **文档** - https://docling-project.github.io/docling/

### 示例实现：Docling

In [ ]:
!pip install docling

In [ ]:
from pathlib import Path
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions

def convert_medium_pdfs_docling(pdf_folder: str, output_folder: str):
    """
    Convert medium-complexity PDFs using Docling with OCR and table extraction.

    Args:
        pdf_folder: Path to folder containing PDF files
        output_folder: Path to output folder for Markdown files
    """
    pipeline_options = PdfPipelineOptions()
    pipeline_options.do_table_structure = True  # Extract table structures
    pipeline_options.do_ocr = True  # Enable OCR for scanned content
    pipeline_options.images_scale = 2.0  # Higher quality image extraction
    pipeline_options.generate_picture_images = True

    converter = DocumentConverter(
        format_options={
            InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)
        }
    )

    output_path = Path(output_folder)
    output_path.mkdir(parents=True, exist_ok=True)

    pdf_files = list(Path(pdf_folder).glob("*.pdf"))

    for pdf_file in pdf_files:
        try:
            result = converter.convert(str(pdf_file))
            markdown_content = result.document.export_to_markdown()

            output_file = output_path / f"{pdf_file.stem}.md"
            output_file.write_text(markdown_content, encoding='utf-8')

            print(f"✓ Converted: {pdf_file.name}")

        except Exception as e:
            print(f"✗ Error processing {pdf_file.name}: {e}")

    print(f"\nConversion complete! Output in '{output_folder}'")

# Example usage
convert_medium_pdfs_docling("./medium_pdfs", "./md_output")

---

## 类别 3：高复杂 PDF - 视觉语言模型（VLM）

**适用场景：** PDF 含有关键视觉信息（图表、示意图、复杂布局或图像密集内容），需要对视觉元素做准确描述与上下文化。

### VLM 方法

VLM 方法的核心是：将 PDF 每页转成高分辨率图片，并配合指令发送给视觉语言模型，输出结构化 Markdown。该方法利用模型的视觉理解能力实现：

1. **识别任意布局和方向的文本**
2. **理解图表、示意图与图片等视觉元素**
3. **识别文档元素之间的空间关系**
4. **通过 Markdown 保留文档结构**
5. **为非文本元素生成语义描述**

**工作流程：**
- 将每页渲染为高分辨率图片（通常 300 DPI）
- 将图片与详细系统提示词一起发送给 VLM
- 模型解析视觉内容并输出结构化 Markdown
- 按页顺序处理并合并为完整文档

该方法强大之处在于模型以“类似人类阅读”的方式理解文档：不仅看文本，还能理解版式和视觉语义。

### 为什么选择 VLM？

- **最佳场景：** 科研论文、信息图、医疗报告、技术图示
- **优点：** 精度高、视觉描述能力强、能保留空间关系、适配复杂布局
- **缺点：** 处理更慢、需要 API 成本（或本地大算力）

### 云端 vs 本地部署

**云端 VLM（大多数用户推荐）：**
- Google Gemini 3 Flash / 2.5 Pro
- OpenAI GPT-4o / GPT-4o-mini
- Anthropic Claude 3.5 Sonnet / Haiku

**本地 VLM（隐私/离线场景）：**
- 性能高度依赖模型规模与硬件
- 模型越大通常精度越高（如 Qwen2-VL 72B > Qwen2-VL 7B）
- 示例：Qwen2-VL、LLaVA、BakLLaVA、CogVLM
- 需要较大显存（7B 模型约 24GB+，70B+ 模型约 80GB+）
- 可通过 Ollama、vLLM 或 Hugging Face Transformers 运行

**成本提示：** 在 PDF 转换任务中，Google Gemini 的性价比通常较好。

### VLM 转换的自定义系统提示词

In [ ]:
# Customize this system prompt based on your PDF type (e.g., academic, technical, legal).
# This template works for 90% of documents—tweak rules as needed for your use case.
SYSTEM_PROMPT = """You are an expert document parser specializing in converting PDF pages to markdown format.

**Your task:**
Extract ALL content from the provided page image and return it as clean, well-structured markdown.

**Text Extraction Rules:**
1. Preserve the EXACT text as written (including typos, formatting, special characters)
2. Maintain the logical reading order (top-to-bottom, left-to-right)
3. Preserve hierarchical structure using appropriate markdown headers (#, ##, ###)
4. Keep paragraph breaks and line spacing as they appear
5. Use markdown lists (-, *, 1.) for bullet points and numbered lists
6. Preserve text emphasis: **bold**, *italic*, `code`
7. For multi-column layouts, extract left column first, then right column

**Tables:**
- Convert all tables to markdown table format
- Preserve column alignment and structure
- Use | for columns and - for headers

**Mathematical Formulas:**
- Convert to LaTeX format: inline `$formula$`, display `$$formula$$`
- If LaTeX conversion is uncertain, describe the formula clearly

**Images, Diagrams, Charts:**
- Insert markdown image placeholder: `![Description](image)`
- Provide a detailed, informative description including:
  * Type of visual (photo, diagram, chart, graph, illustration)
  * Main subject or purpose
  * Key elements, labels, or data points
  * Colors, patterns, or notable visual features
  * Context or relationship to surrounding text
- For charts/graphs: mention axes, data trends, and key values
- For diagrams: describe components and their relationships

**Special Elements:**
- Footnotes: Use markdown footnote syntax `[^1]`
- Citations: Preserve as written
- Code blocks: Use triple backticks with language specification
- Quotes: Use `>` for blockquotes
- Links: Preserve as `[text](url)` if visible

**Quality Guidelines:**
- DO NOT add explanations, comments, or meta-information
- DO NOT skip or summarize content
- DO NOT invent or hallucinate text not present in the image
- DO NOT include "Here is the markdown..." or similar preambles
- Output ONLY the markdown content, nothing else

**Output Format:**
Return raw markdown with no wrapper, no code blocks, no explanations.
Start immediately with the page content.
""".strip()

### 使用 Google Gemini 的实现

该实现演示了“逐页转换”的处理方式。

**参考：** [Gemini API – Image Understanding](https://ai.google.dev/gemini-api/docs/image-understanding)



In [ ]:
!pip install PyMuPDF google-genai

In [ ]:
import os
import fitz  # PyMuPDF
from google import genai
from google.genai import types

def convert_complex_pdfs_vlm(pdf_path: str, api_key: str, model: str = "gemini-2.5-flash"):
    """
    Convert a single PDF using VLM (Vision-Language Model).

    Args:
        pdf_path: Path to PDF file
        api_key: Google Gemini API key
        model: Model name (gemini-2.5-flash, gemini-2.5-pro, etc.)

    Returns:
        Dictionary mapping page numbers to markdown content
    """
    client = genai.Client(api_key=api_key)
    pdf_document = fitz.open(pdf_path)
    markdown_pages = {}

    for page_num in range(pdf_document.page_count):
        try:
            page = pdf_document[page_num]

            # Convert page to high-resolution image (300 DPI)
            pix = page.get_pixmap(matrix=fitz.Matrix(300/72, 300/72))
            img_data = pix.tobytes("png")

            # Prepare image for VLM
            image = types.Part.from_bytes(data=img_data, mime_type="image/png")

            # Generate markdown from image
            response = client.models.generate_content(
                config=types.GenerateContentConfig(
                    system_instruction=SYSTEM_PROMPT,
                    temperature=0.1  # Low temperature for consistent output
                ),
                model=model,
                contents=[
                    "Convert this PDF page to clean, structured markdown. "
                    "Extract all text, describe images, and preserve the layout.",
                    image
                ],
            )

            markdown_pages[page_num + 1] = response.text
            print(f"✓ Processed page {page_num + 1}/{pdf_document.page_count}")

        except Exception as e:
            print(f"✗ Error on page {page_num + 1}: {e}")
            markdown_pages[page_num + 1] = f"<!-- Error processing page: {e} -->"

    pdf_document.close()
    return markdown_pages


def batch_convert_complex_pdfs(pdf_folder: str, output_folder: str, api_key: str):
    """
    Batch convert all PDFs in a folder using VLM.

    Args:
        pdf_folder: Path to folder containing PDF files
        output_folder: Path to output folder for Markdown files
        api_key: Google Gemini API key
    """
    os.makedirs(output_folder, exist_ok=True)

    for filename in os.listdir(pdf_folder):
        if filename.lower().endswith('.pdf'):
            print(f"\n📄 Processing: {filename}")
            pdf_path = os.path.join(pdf_folder, filename)
            pdf_name = os.path.splitext(filename)[0]

            # Convert PDF
            markdown_pages = convert_complex_pdfs_vlm(pdf_path, api_key)

            # Combine pages into single markdown file
            combined_markdown = "\n\n---\n\n".join([
                f"# Page {page_num}\n\n{content}"
                for page_num, content in markdown_pages.items()
            ])

            # Save to file
            output_path = os.path.join(output_folder, f"{pdf_name}.md")
            with open(output_path, 'w', encoding='utf-8') as f:
                f.write(combined_markdown)

            print(f"✓ Saved: {output_path}")

    print(f"\n🎉 Batch conversion complete! Output in '{output_folder}'")

# Example usage
batch_convert_complex_pdfs("./complex_pdfs", "./md_output", "your-gemini-api-key")

## 推荐工作流

```
┌─────────────────────────────┐
│  分析 PDF 文档              │
│  1. 是否为扫描件？          │
│  2. 图片是否关键？          │
│  3. 布局是否复杂？          │
│  4. 是否科学内容？          │
└──────────┬──────────────────┘
           │
           ▼
    ┌──────────────────┐
    │ 数字原生且简单？ │───Yes──► PyMuPDF4LLM
    └──────┬───────────┘
           │No
           ▼
    ┌────────────────────────────────┐
    │ 扫描件 / 仅表格                │
    │ （无关键视觉内容）？           │───Yes──► Docling
    └──────┬─────────────────────────┘
           │No
           ▼
    ┌──────────────────────────────┐
    │ 复杂布局、图表、示意图或      │
    │ 需要视觉理解的内容？          │───Yes──► VLM (Gemini)
    └──────────────────────────────┘
```

---

## 扫描文档的特殊注意事项

**重要：** 扫描 PDF 无论布局复杂与否都必须使用 OCR。即使版式简单，只要文本不可选中，也应使用类别 2 工具（Docling、Marker、PaddleOCR）。

**如何识别扫描 PDF：**
1. 在 PDF 查看器中尝试选中文本，若无法选中，通常是扫描件
2. 查看文件大小，扫描件通常更大
3. 观察是否存在图像伪影或轻微倾斜文字

**按场景推荐工具：**
- **英文扫描文档：** Docling 或 Marker
- **多语言扫描文档：** PaddleOCR（支持 80+ 语言）
- **低质量扫描：** VLM 方法通常精度更高
- **大批量扫描文档：** Marker（处理速度通常更快）

---

## 最佳实践

### 1. 先在样本文档上测试
在处理整库文档前，先用每类代表样本验证抽取质量。

### 2. 增加质量检查

In [ ]:
def validate_markdown_quality(md_file: Path) -> dict:
    """Check markdown conversion quality"""
    content = md_file.read_text()
    words = content.split()

    return {
        "word_count": len(words),
        "has_headers": "#" in content,
        "has_tables": "|" in content,
        "has_formulas": "$" in content,
        "avg_line_length": len(content) / max(content.count("\n"), 1),
        "empty_ratio": content.count("\n\n") / max(content.count("\n"), 1)
    }

# Example usage
quality_metrics = validate_markdown_quality(Path("output.md"))
print(f"Quality metrics: {quality_metrics}")

### 3. 处理混合文档类型

In [ ]:
def smart_convert_pdf(pdf_path: str, api_key: str = None):
    """
    Intelligently choose conversion method based on PDF characteristics.
    """
    # Quick analysis
    doc = fitz.open(pdf_path)
    sample_page = doc[0]

    # Check if text is selectable
    text = sample_page.get_text()
    is_scanned = len(text.strip()) < 50  # Likely scanned if very little text

    # Check for images
    image_count = len(sample_page.get_images())
    has_images = image_count > 2

    doc.close()

    # Route to appropriate tool
    if is_scanned:
        print("→ Using Docling (scanned document)")
        return convert_medium_pdfs_docling(pdf_path, "output")
    elif has_images:
        print("→ Using VLM (image-heavy)")
        return convert_complex_pdfs_vlm(pdf_path, api_key)
    else:
        print("→ Using PyMuPDF4LLM (simple digital PDF)")
        return convert_simple_pdfs_pymupdf4llm(pdf_path, "output")

---

## 总结

选择合适的 PDF-to-Markdown 转换器会直接影响 RAG 系统性能。

**核心原则：**
1. 先用能满足需求的最简单工具
2. 仅在质量不足时升级到更复杂方案
3. 混合文档场景采用混合策略
4. 全量部署前务必验证输出质量
5. 生产环境持续监控成本与性能

对于生产系统，建议实现“智能路由”机制：根据文档特征自动选择最佳转换工具。这样可以同时兼顾质量与成本效率，并在整个文档语料上保持稳定结果。

**下一步建议：**
- 在你的真实文档类型上测试不同工具
- 量化质量、速度与成本之间的权衡
- 迭代优化 VLM 的系统提示词
- 建立自动化质量校验流水线
- 在持续监控性能的前提下逐步扩容

## 其他 PDF 抽取与处理工具

- **PDFPlumber** – https://github.com/jsvine/pdfplumber
- **MarkItDown** – https://github.com/microsoft/markitdown
- **Marker** – https://github.com/VikParuchuri/marker
- **Unstructured** – https://github.com/Unstructured-IO/unstructured
- **DeepSeek OCR** – https://github.com/deepseek-ai/DeepSeek-OCR
- **DeepSeek OCR 2** - https://github.com/deepseek-ai/DeepSeek-OCR-2
- **Paddle OCR** – https://github.com/PaddlePaddle/PaddleOCR
- **Surya OCR** – https://github.com/VikParuchuri/surya
- **Dolphin** – https://github.com/bytedance/Dolphin
- **LlamaParse** – https://github.com/run-llama/llama_parse
- **Nougat** – https://github.com/facebookresearch/nougat
- **MinerU** – https://github.com/opendatalab/MinerU
- **Dots.ocr** - https://github.com/rednote-hilab/dots.ocr
- **GLM OCR** - https://github.com/zai-org/GLM-OCR
- **Liteparse** - https://github.com/run-llama/liteparse